# RAG & LCEL Mastery: A Future-Proof Guide

Welcome to this comprehensive guide on **Retrieval-Augmented Generation (RAG)** and **LangChain Expression Language (LCEL)**.

Whether you are revisiting this one year from now or learning it for the first time, this notebook is designed to explain *why* we do things, not just *how*.

## Objectives
1.  **Ingest** a blog post into a knowledge base.
2.  **Understand** the "magic" behind Text Splitting and Embeddings.
3.  **Build** a RAG pipeline to answer questions about the blog.
4.  **Master LCEL**: Move from procedural code to declarative, production-ready chains.
5.  **Deep Dive**: Understand difficult concepts like `RunnablePassthrough`, `itemgetter`, and the pipe `|` operator.

## 1. Setup & Environment

First, let's setup out environment. We need to load our API keys (Google Gemini & Pinecone) from the `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify keys are present (optional check)
if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY not found in .env")
if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY not found in .env")

print("Environment loaded successfully!")

## Part 1: Ingestion (ETL for AI)

**Ingestion** is the process of taking unstructured data (like our blog post) and preparing it for the LLM. LLMs don't know your private data; we need to "teach" them by providing context.

### Step 1.1: Load Data
We use `TextLoader` to read the file.

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./meduim_blog.txt")
document = loader.load()

print(f"Loaded document with {len(document[0].page_content)} characters.")

### Step 1.2: Splitting (The Art of Chunking)

**Why split?**
1.  **Context Limits**: LLMs have a limit on how much text they can process.
2.  **Semantic Relevance**: If you search for "apple", you want the paragraph about apples, not the entire book.

#### 🆚 Deep Dive: `RecursiveCharacterTextSplitter` vs `CharacterTextSplitter`

This is a common point of confusion. Let's clarify.

**1. CharacterTextSplitter (The "Dumb" Splitter)**
- **How it works**: It splits strictly based on a single character (usually `\n\n` or just empty space).
- **Problem**: It doesn't care about the structure of language. It might cut a sentence in half if the chunk size limit is reached.
- **Use Case**: Simple text where structure doesn't matter much.

**2. RecursiveCharacterTextSplitter (The "Smart" Splitter)**
- **How it works**: It tries to split recursively using a list of separators in order: `["\n\n", "\n", " ", ""]`.
- **Logic**: 
    1.  Can I split by paragraphs (`\n\n`)? Yes? Great. 
    2.  Is the chunk still too big? Okay, try splitting by lines (`\n`).
    3.  Still too big? Try splitting by words (` `).
- **Benefit**: It preserves semantic meaning by keeping related text (paragraphs/sentences) together.

**Recommendation**: Always default to `RecursiveCharacterTextSplitter` for natural language.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Target size of each chunk
    chunk_overlap=200,    # Overlap to keep context between chunks
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.split_documents(document)
print(f"Split into {len(texts)} chunks.")

# Let's look at the first chunk content
print("\n--- First Chunk ---\n")
print(texts[0].page_content[:200] + "...")

### Step 1.3: Embeddings & Vector Store

**Embeddings**: Converting text into a list of numbers (vectors). Similar text = Similar numbers.
**Vector Store**: A database optimized to search these numbers fast (Pinecone).

We use `GoogleGenerativeAIEmbeddings`.

In [ ]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Create/Update the Vector Store
# Note: This actually sends data to Pinecone. 
# If you've already run this, you can skip .from_documents and just initialize existing store.
vectorstore = PineconeVectorStore.from_documents(
    texts, 
    embeddings, 
    index_name=os.getenv("INDEX_NAME")
)

print("Successfully stored vectors in Pinecone!")

## Part 2: Retrieval

Now that data is in Pinecone, we need a way to fetch it.
We convert our `vectorstore` into a `retriever`.

- `k=3`: Retrieve the top 3 most similar chunks.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Helper function to format retrieved documents into a single string
def format_documents(documents):
    return "\n".join([doc.page_content for doc in documents])

## Part 3: Generation (The Model)

We setup our LLM (Gemini) and the Prompt Template.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

# Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5,
    max_retries=2,
)

# Setup Prompt
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("human", """Answer the question based on the context provided.
    Context: {context}

    Question: {question}

    Provide a detailed answer.""")
    ]
)

## Part 4: The Chain (Where the Magic Happens)

This is the most critical part to understand. We will solve this in two ways.

### Approach A: The "Hard Way" (Without LCEL)

This is standard Python procedural code. It's explicit but verbose.

In [ ]:
def retrieval_chain_without_lcel(query: str):
    print("--- Executing WITHOUT LCEL ---")
    
    # 1. Retrieve relevant docs
    print("1. Retrieving documents...")
    docs = retriever.invoke(query)
    
    # 2. Format them
    print("2. Formatting documents...")
    formatted_docs = format_documents(docs)
    
    # 3. Create the prompt with variables filled
    print("3. Creating prompt...")
    prompt = prompt_template.format_messages(context=formatted_docs, question=query)
    
    # 4. Invoke the LLM
    print("4. Generating response...")
    response = llm.invoke(prompt)
    
    return response.content

### Approach B: The "Smart Way" (With LCEL)

**LangChain Expression Language (LCEL)** allows us to chain components using the `|` pipe operator.

#### 🛑 The Problem: Piping Dictionaries vs Strings
Our input is usually a dictionary: `{"question": "What is RAG?"}`.
We need to pass `context` and `question` to the prompt.

#### 🤯 Deep Dive: `RunnablePassthrough.assign`

Ideally, we want to add the `context` to our input dictionary, so it looks like:
`{"question": "...", "context": "..."}`

- `RunnablePassthrough` allows the input to pass specific steps unchanged.
- `.assign(...)` is like `dict.update()`. It calculates a new value and adds it to the stream.

#### 🤯 Deep Dive: `itemgetter` vs `lambda`

You might see `itemgetter("question")` used often. Why not `lambda x: x["question"]`?

The pipe `|` operator requires **Runnables** (objects with an `.invoke()` method) on the left and right.
- **WRONG**: `lambda x: x["question"] | retriever`. Inside the lambda, `x["question"]` is a **STRING**. Strings don't have a `|` operator for retrievers.
- **RIGHT**: `itemgetter("question") | retriever`. `itemgetter` returns a Callable, which LangChain wraps as a Runnable. It correctly passes the output to the retriever.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

def retrieval_chain_with_lcel():
    retrieval_chain = (
        # 1. Take input {"question": "..."} and add "context" key
        RunnablePassthrough.assign(
            # Get the question -> Retrieve docs -> Format them
            context=itemgetter("question") | retriever | format_documents
        )
        # 2. Now input is {"question": "...", "context": "..."}
        # Pass it to the prompt template
        | prompt_template
        # 3. Pass populated prompt to LLM
        | llm
        # 4. Parse the output (convert AI Message to String)
        | StrOutputParser()
    )
    
    return retrieval_chain

## 5. Execution & Verification

Let's test both methods.

In [ ]:
query = "what is RAG and what are the challengs in RAG ?"

# Test 1: Without LCEL
print("\n" + "="*40)
print("Testing WITHOUT LCEL")
print("="*40)
response1 = retrieval_chain_without_lcel(query)
print("\nANSWER:\n", response1)


# Test 2: With LCEL
print("\n" + "="*40)
print("Testing WITH LCEL")
print("="*40)
chain = retrieval_chain_with_lcel()
# Notice we pass a dictionary, because itemgetter expects a key lookup
response2 = chain.invoke({"question": query})
print("\nANSWER:\n", response2)

## Summary

Congratulations! You've built a RAG pipeline and mastered some advanced LangChain concepts.

**Key Takeaways:**
1.  **Ingest carefully**: Use `RecursiveCharacterTextSplitter` to keep distinct thoughts together.
2.  **LCEL is powerful**: The pipe `|` operator makes readable, streamable chains, but remember you can only pipe **Runnables**.
3.  **Use `itemgetter`**: When grabbing data from a dictionary in a chain, it's safer and cleaner than lambdas for piping.